# mask meta：G × 分块一阶内积 金标准
**回答的问题：G = |∂L_task/∂x₀| 作为掩码选址代理有没有价值。**

空间版"ut × meta-grad"：真实量是**分块一阶内积** IP_i = Σ_{p∈i} g_task(p)·g_den(p)
（输入空间分解，Σ_i IP_i = ⟨∂L_task/∂x₀, ∂L_den/∂x₀⟩，与论文 3.3 的一阶展开逐块对应）；
代理量是 G 的分块均值（与在线 MaskSelector 同口径）。

每个 ckpt 出一张**五联图**：原图 | IP 正部热力 | G 热力 | **IP 掩码软叠加** | **G 掩码软叠加**。
两个掩码由两个概率场（softmax(IP 正部)、softmax(G)）经**同一组 Gumbel 噪声**无放回定量采出
（CRN 配对：采样随机性被消去，蒙区差异只反映概率场差异）；软叠加 = 模型实际看到的退化输入
x̃₀ = w⊙x₀+(1−w)⊙c，w = 1−α(1−m)，c 为该图均值色（与训练 MaskApplier 同公式，α 可调）。
每行标注逐图 Spearman ρ / Pearson r 与**两掩码蒙区的块级 IoU**——
即使逐块相关受 IP 噪声影响波动，top-k 选址重合度（IoU）才是部署时真正起作用的量。
跨 ckpt 汇总 ρ / r / IoU 演化；测量结果缓存 npz，复画免重测。
CRN：跨 ckpt 同图、同 t、同噪声，严格可比。

**波动说明**：逐图 ρ 的跨图 std 大 = 图像间真实差异（语义结构不同）+ IP 估计噪声（t/ε 采样）
两部分叠加；增大 NUM_IMAGES 只压统计误差，增大 N_T / N_NOISE 只压 IP 估计噪声，
图像间真实差异是现象本身，不是缺陷。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把两个路径都算成绝对路径 ----
VISUAL = os.path.abspath('.')                                  # 内核启动时 cwd 就是 visual/
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAE）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import probe_core as pc

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CFG 用 run 目录里 train 时 dump 的完整 config；CKPTS 按 iter 排序。
# 旧 run 兼容：build_model 会自动剔除 dump config 里已退役的机制键，旧 ckpt 直接可测；
# UNet 仪器的旧 run 需把 UNET_FIX 改为 dict(denoiser_type='unet')。
import glob, re
RUN_DIR = 'work_dirs/eden/fcn_diffae'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]   # run 目录里 dump 的 config
UNET_FIX = None   # UNet 仪器的旧 run 改为 dict(denoiser_type='unet')；DiT run 保持 None
CKPTS = sorted(glob.glob(f'{RUN_DIR}/iter_*.pth'),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = 'work_dirs/offline_probe/mask_meta'
NUM_IMAGES = 8        # 统计用图数（相关性/IoU 均值±std 的样本量；跨 ckpt 固定）
DISPLAY_IMAGES = 4    # 五联图只画前几张（统计仍用全部 NUM_IMAGES）
MASK_RATIO = None     # 双掩码的蒙块量；None = 模型配置值（旧 run ratio=0 时回退 0.25）
MASK_ALPHA = 1.0      # 软叠加程度：1=蒙区完全替换为该图均值色，0=不蒙（仅展示，同训练 MaskApplier 公式）
N_T = 8               # IP 测量的 t 点数（训练网格等距；调大可压 IP 估计噪声，成本线性）
N_NOISE = 2           # 每 t 的 CRN 噪声数
SEED = 0
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG, cfg_overrides=UNET_FIX)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
rows = []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt, cfg_overrides=UNET_FIX)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    res = pc.measure_mask_meta(model, batch, n_t=N_T, n_noise=N_NOISE,
                               mask_ratio=MASK_RATIO, seed=SEED)
    # 缓存（复画免重测；img 一并存，供免重测复画软叠加）
    import numpy as np
    np.savez(os.path.join(OUT, f'{tag}_cache.npz'),
             img=res['img'].cpu().numpy(),
             g_block=res['g_block'].cpu().numpy(),
             ip_block=res['ip_block'].cpu().numpy(),
             mask=res['mask'].cpu().numpy(),
             mask_ip=res['mask_ip'].cpu().numpy(),
             rho_s=res['rho_s'], rho_p=res['rho_p'], iou=res['iou'], t=res['t'])
    fig = pc.plot_mask_meta(res, title=tag, max_images=DISPLAY_IMAGES,
                            mask_alpha=MASK_ALPHA,
                            save_path=os.path.join(OUT, f"{tag}_mask_meta.png"))
    rows.append((tag, res['rho_s'], res['rho_p'], res['iou']))
    print(f"{tag}: Spearman ρ {res['rho_s'].mean():.3f}±{res['rho_s'].std():.3f} | "
          f"Pearson r {res['rho_p'].mean():.3f}±{res['rho_p'].std():.3f} | "
          f"mask IoU {res['iou'].mean():.3f}±{res['iou'].std():.3f}")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 汇总：G ~ 分块一阶内积 的相关性 + 掩码选址 IoU 随训练演化（均值 ± 跨图 std）
# IoU 是部署口径：即使逐块相关波动，top-k 选址重合才是掩码机制实际吃到的东西
import numpy as np
import matplotlib.pyplot as plt
if len(rows) > 1:
    tags = [r[0] for r in rows]
    series = [('Spearman ρ', np.array([r[1] for r in rows]), 'tab:blue'),
              ('Pearson r', np.array([r[2] for r in rows]), 'tab:orange'),
              ('mask IoU', np.array([r[3] for r in rows]), 'tab:green')]
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for name, vals, c in series:
        m, s = vals.mean(1), vals.std(1)
        ax.plot(tags, m, 'o-', color=c, label=name)
        ax.fill_between(tags, m - s, m + s, color=c, alpha=0.15)
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.set_ylim(-1, 1); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.set_title('G ~ block-wise meta IP & mask IoU across checkpoints (mean ± std over images)')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.savefig(os.path.join(OUT, 'summary.png'), dpi=150, bbox_inches='tight')
    fig

In [ ]:
# 免重测复画：从 npz 缓存重建 res，改 MASK_RATIO 不行（掩码已在缓存里）、
# 但 MASK_ALPHA / DISPLAY_IMAGES / 标题都能直接换
import numpy as np
REPLAY_TAG = CKPTS[-1] and os.path.splitext(os.path.basename(CKPTS[-1]))[0]  # 改成想复画的 ckpt tag
z = np.load(os.path.join(OUT, f'{REPLAY_TAG}_cache.npz'))
res = {'img': torch.from_numpy(z['img']),
       'g_block': torch.from_numpy(z['g_block']),
       'ip_block': torch.from_numpy(z['ip_block']),
       'mask': torch.from_numpy(z['mask']),
       'mask_ip': torch.from_numpy(z['mask_ip']),
       'rho_s': z['rho_s'], 'rho_p': z['rho_p'], 'iou': z['iou']}
fig = pc.plot_mask_meta(res, title=f'{REPLAY_TAG} (replay, α={MASK_ALPHA:g})',
                        max_images=DISPLAY_IMAGES, mask_alpha=MASK_ALPHA,
                        save_path=os.path.join(OUT, f"{REPLAY_TAG}_replay.png"))
fig